# Explore here

In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# API de Keras/TensorFlow para cargar imágenes y construir la RNA.
try:
    from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
    from tensorflow.keras.applications import EfficientNetB0
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
    from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
    from tensorflow.keras.models import Sequential, load_model
except ImportError as exc:
    raise ImportError(
        "Este ejercicio requiere tensorflow. Instálalo con `pip install tensorflow` "
        "y vuelve a ejecutar el notebook."
    ) from exc

ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
SOURCE_TRAIN_DIR = ROOT / "train"
IMAGE_SIZE = (200, 200)
MODEL_IMAGE_SIZE = (224, 224)


def image_paths_for(label, n=9):
    return sorted(SOURCE_TRAIN_DIR.glob(f"{label}.*"))[:n]


def show_images(paths, title):
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, path in zip(axes.flat, paths):
        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(path.name)
        ax.axis("off")
    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    plt.show()


show_images(image_paths_for("dog"), "Primeras nueve fotos de perros")
show_images(image_paths_for("cat"), "Primeras nueve fotos de gatos")


def total_ram_gb():
    if hasattr(os, "sysconf"):
        pages = os.sysconf("SC_PHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        return pages * page_size / (1024**3)
    return 0.0


all_image_paths = sorted(
    path
    for path in SOURCE_TRAIN_DIR.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

if total_ram_gb() > 12:
    photos = np.asarray(
        [img_to_array(load_img(path, target_size=IMAGE_SIZE, color_mode="rgb")) for path in all_image_paths],
        dtype=np.uint8,
    )
    labels = np.asarray(
        [1 if path.name.lower().startswith("dog.") else 0 for path in all_image_paths],
        dtype=np.uint8,
    )
    processed_data = (photos, labels)
    print(f"processed_data: fotos={photos.shape}, etiquetas={labels.shape}")
else:
    processed_data = None
    print(f"RAM total: {total_ram_gb():.1f} GB. Carga progresiva activada.")

# sampleSubmission.csv no contiene imágenes. Se crea un test estratificado del
# 20 % del train original con enlaces simbólicos, sin duplicar archivos.
IMAGE_DATA_DIR = ROOT / "data" / "image_data"
RNG = np.random.default_rng(42)
for class_name, prefix in (("cats", "cat."), ("dogs", "dog.")):
    class_paths = sorted(SOURCE_TRAIN_DIR.glob(f"{prefix}*"))
    shuffled = class_paths.copy()
    RNG.shuffle(shuffled)
    split_index = int(len(shuffled) * 0.8)
    for split, paths in (("train", shuffled[:split_index]), ("test", shuffled[split_index:])):
        destination_dir = IMAGE_DATA_DIR / split / class_name
        destination_dir.mkdir(parents=True, exist_ok=True)
        for source_path in paths:
            destination = destination_dir / source_path.name
            if not destination.exists():
                destination.symlink_to(source_path.resolve())

train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)
test_datagen = ImageDataGenerator()

trdata = train_datagen.flow_from_directory(
    IMAGE_DATA_DIR / "train",
    target_size=MODEL_IMAGE_SIZE,
    batch_size=32,
    class_mode="categorical",
    classes=["cats", "dogs"],
    shuffle=True,
    seed=42,
)
tsdata = test_datagen.flow_from_directory(
    IMAGE_DATA_DIR / "test",
    target_size=MODEL_IMAGE_SIZE,
    batch_size=32,
    class_mode="categorical",
    classes=["cats", "dogs"],
    shuffle=False,
)

print("Clases detectadas:", trdata.class_indices)
print("Imágenes de entrenamiento:", trdata.samples)
print("Imágenes de prueba:", tsdata.samples)

# -----------------------------------------------------------------------------
# RNA EfficientNet-B0 y optimización mediante callbacks
# -----------------------------------------------------------------------------
model = Sequential(name="efficientnet_b0_cat_dog")
model.add(EfficientNetB0(include_top=False, weights=None, input_shape=(224, 224, 3)))
model.add(GlobalAveragePooling2D())
model.add(Dense(units=128, activation="relu"))
model.add(Dense(units=2, activation="softmax"))
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
BEST_MODEL_PATH = MODEL_DIR / "efficientnet_b0_best.keras"

checkpoint = ModelCheckpoint(
    filepath=BEST_MODEL_PATH,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1,
)
early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=3,
    restore_best_weights=False,
    verbose=1,
)
callbacks = [checkpoint, early_stopping]

# fit_generator fue eliminado en versiones modernas de Keras; model.fit()
# acepta exactamente el mismo generador y los mismos callbacks.
history = model.fit(
    trdata,
    validation_data=tsdata,
    epochs=10,
    callbacks=callbacks,
)

# Cargar explícitamente el mejor checkpoint, no necesariamente la última época.
best_model = load_model(BEST_MODEL_PATH)
test_loss, test_accuracy = best_model.evaluate(tsdata, verbose=1)
print(f"Pérdida del mejor modelo en test: {test_loss:.4f}")
print(f"Precisión del mejor modelo en test: {test_accuracy:.4f}")

# Predicciones del mejor modelo. Como tsdata no mezcla las imágenes, el orden
# coincide con tsdata.filepaths.
tsdata.reset()
prediction_probabilities = best_model.predict(tsdata, verbose=1)
predicted_indices = np.argmax(prediction_probabilities, axis=1)
index_to_class = {index: name for name, index in tsdata.class_indices.items()}
predicted_labels = np.asarray(
    [index_to_class[index] for index in predicted_indices],
    dtype=str,
)

predictions = {
    "filepaths": np.asarray(tsdata.filepaths),
    "class_indices": predicted_indices,
    "labels": predicted_labels,
    "probabilities": prediction_probabilities,
}
print("Predicciones generadas:", len(predicted_labels))
print(predictions["labels"][:10])

# Curvas de aprendizaje.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="test")
axes[0].set_title("Precisión")
axes[0].set_xlabel("Época")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="test")
axes[1].set_title("Pérdida")
axes[1].set_xlabel("Época")
axes[1].legend()
fig.tight_layout()
plt.show()


In [ ]:
# Guardado final del modelo en la carpeta models/.
# El checkpoint ya conserva el mejor modelo según val_accuracy; aquí se copia
# explícitamente con un nombre estable para reutilizarlo posteriormente.
FINAL_MODEL_PATH = MODEL_DIR / "efficientnet_b0_cat_dog.keras"
best_model.save(FINAL_MODEL_PATH)
print(f"Modelo guardado en: {FINAL_MODEL_PATH}")
